# Multi-Sleeve US Equities Statistical Arbitrage

Fully self-contained reproduction of key results from the multi-sleeve stat-arb paper.
No imports from `qstudy` or any local scripts — everything is inlined.

**Sections:**
0. Imports & Configuration  
1. Data Download  
2. Inlined Pipeline Infrastructure  
3. Run All 8 Sleeves (IS period)  
4. Figure 1 — Sleeve Return Correlation Heatmap  
5. Table 3 — Sleeve-by-Sleeve Portfolio Buildout  
6. Figure 2 — Robustness Heatmap  
7. Table 5 — Equal-Vol Attribution  
8. Figure 3 — Transaction Cost Sensitivity  
9. Figure 4 — Target-Vol Equity Curves  
10. Table 7 + Figure 5 — Full History OOS Analysis (2015–2026)


## Section 0: Imports & Configuration


In [ ]:
import os
import json
import warnings
import itertools
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import yfinance as yf
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
matplotlib.rcParams.update({"figure.dpi": 120, "font.size": 10})

# ── Output directory ───────────────────────────────────────────────────────────
OUT_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "out")
os.makedirs(OUT_DIR, exist_ok=True)

# ── Constants ──────────────────────────────────────────────────────────────────
START_DATE  = "2015-01-01"
IS_END      = "2023-12-31"
OOS_END     = "2026-04-30"
OOS_SPLIT   = pd.Timestamp("2024-01-01")
BENCHMARK   = "SPY"
FACTORS     = ["SPY", "XLK", "XLF", "XLE", "XLV", "XLI", "XLY", "XLP", "XLU", "XLRE", "XLB"]
SECTOR_ETFS = ["XLK", "XLF", "XLE", "XLV", "XLI", "XLY", "XLP", "XLU", "XLRE", "XLB"]
COST_BPS    = 10.0
N_LONG      = 20
N_SHORT     = 20

GICS_TO_ETF = {
    "Technology": "XLK", "Financial Services": "XLF", "Energy": "XLE",
    "Healthcare": "XLV", "Industrials": "XLI", "Consumer Cyclical": "XLY",
    "Consumer Defensive": "XLP", "Utilities": "XLU", "Real Estate": "XLRE",
    "Basic Materials": "XLB", "Communication Services": "XLK",
    "Financial": "XLF", "Consumer Staples": "XLP", "Materials": "XLB",
    "Information Technology": "XLK",
}

SP500 = [
    "MMM","AOS","ABT","ABBV","ACN","ADBE","AMD","AES","AFL","A","APD","ABNB","AKAM","ALB",
    "ARE","ALGN","ALLE","LNT","ALL","GOOGL","GOOG","MO","AMZN","AMCR","AEE","AAL","AEP",
    "AXP","AIG","AMT","AWK","AMP","AME","AMGN","APH","ADI","ANSS","AON","APA","AAPL",
    "AMAT","APTV","ACGL","ADM","ANET","AJG","AIZ","T","ATO","ADSK","AZO","AVB","AVY",
    "AXON","BKR","BALL","BAC","BBWI","BAX","BDX","WRB","BBY","BIO","TECH","BIIB","BLK",
    "BX","BA","BCH","BMY","AVGO","BR","BRO","BF.B","BLDR","BG","CDNS","CZR","CPT","CPB",
    "COF","CAH","KMX","CCL","CARR","CTLT","CAT","CBOE","CBRE","CDW","CE","COR","CNC",
    "CNX","CDAY","CF","CRL","SCHW","CHTR","CVX","CMG","CB","CHD","CI","CINF","CTAS",
    "CSCO","C","CFG","CLX","CME","CMS","KO","CTSH","CL","CMCSA","CMA","CAG","COP","ED",
    "STZ","CEG","COO","CPRT","GLW","CTVA","CSGP","COST","CTRA","CCI","CSX","CMI","CVS",
    "DHI","DHR","DRI","DVA","DE","DAL","XRAY","DVN","DXCM","FANG","DLR","DFS","DG",
    "DLTR","D","DPZ","DOV","DOW","DTE","DUK","DD","EMN","ETN","EBAY","ECL","EIX","EW",
    "EA","ELV","LLY","EMR","ENPH","ETR","EOG","EPAM","EQT","EFX","EQIX","EQR","ESS",
    "EL","ETSY","EG","EVRG","ES","EXC","EXPE","EXPD","EXR","XOM","FFIV","FDS","FICO",
    "FAST","FRT","FDX","FIS","FITB","FSLR","FE","FI","FMC","F","FTNT","FTV","FOXA",
    "FOX","BEN","FCX","GRMN","IT","GEHC","GEN","GNRC","GD","GE","GIS","GM","GPC","GILD",
    "GPN","GL","GS","HAL","HIG","HAS","HCA","DOC","HSIC","HSY","HES","HPE","HLT","HOLX",
    "HD","HON","HRL","HST","HWM","HPQ","HUBB","HUM","HBAN","HII","IBM","IEX","IDXX",
    "ITW","INCY","IR","PODD","INTC","ICE","IFF","IP","IPG","INTU","ISRG","IVZ","INVH",
    "IQV","IRM","JBHT","JKHY","J","JNJ","JCI","JPM","JNPR","K","KVUE","KDP","KEY",
    "KEYS","KMB","KIM","KMI","KLAC","KHC","KR","LHX","LH","LRCX","LW","LVS","LDOS",
    "LEN","LNC","LIN","LYV","LKQ","LMT","L","LOW","LYB","MTB","MRO","MPC","MKTX","MAR",
    "MMC","MLM","MAS","MA","MTCH","MKC","MCD","MCK","MDT","MRK","META","MET","MTD",
    "MGM","MCHP","MU","MSFT","MAA","MRNA","MHK","MOH","TAP","MDLZ","MPWR","MNST","MCO",
    "MS","MOS","MSI","MSCI","NDAQ","NTAP","NFLX","NWL","NEM","NWSA","NWS","NEE","NKE",
    "NI","NDSN","NSC","NTRS","NOC","NCLH","NRG","NUE","NVDA","NVR","NXPI","ORLY","OXY",
    "ODFL","OMC","ON","OKE","ORCL","OTIS","PCAR","PKG","PANW","PH","PAYX","PAYC","PYPL",
    "PNR","PEP","PFE","PCG","PM","PSX","PNW","PXD","PNC","POOL","PPG","PPL","PFG","PG",
    "PGR","PLD","PRU","PEG","PTC","PSA","PHM","QRVO","PWR","QCOM","DGX","RL","RJF",
    "RTX","O","REG","REGN","RF","RSG","RMD","RVTY","ROK","ROL","ROP","ROST","RCL",
    "SPGI","CRM","SBAC","SLB","STX","SRE","NOW","SHW","SPG","SWKS","SJM","SNA","SOLV",
    "SO","LUV","SWK","SBUX","STT","STLD","STE","SYK","SMCI","SYF","SNPS","SYY","TMUS",
    "TROW","TTWO","TPR","TRGP","TGT","TEL","TDY","TFX","TER","TSLA","TXN","TXT","TMO",
    "TJX","TSCO","TT","TDG","TRV","TRMB","TFC","TYL","TSN","USB","UDR","ULTA","UNP",
    "UAL","UPS","URI","UNH","UHS","VLO","VTR","VRSN","VRSK","VZ","VRTX","VLTO","VMC",
    "WRK","WAB","WBA","WMT","DIS","WBD","WM","WAT","WEC","WFC","WELL","WST","WDC",
    "WHR","WMB","WTW","GWW","WYNN","XEL","XYL","YUM","ZBRA","ZBH","ZTS",
]

SLEEVE_CONFIGS = [
    (
        "factor_model_resid_mr_5d__r10__trend_20_100__cond__residual_dispersion_high_20_q75",
        "factor_model_resid_mr_5d", 10, "trend_20_100",
        "residual_dispersion_high_20_q75", True, False,
    ),
    (
        "mr_5d__r10__trend_50_200__cond__breadth_weak_40",
        "mr_5d", 10, "trend_50_200", "breadth_weak_40", False, False,
    ),
    (
        "dist_mr_k3_z20__r21__cond__vol_contraction_10_60",
        "dist_mr_k3_z20", 21, "none", "vol_contraction_10_60", False, False,
    ),
    (
        "dist_mr_k3_z60__r5__cond__vol_expansion_10_60",
        "dist_mr_k3_z60", 5, "none", "vol_expansion_10_60", False, False,
    ),
    (
        "dist_mr_k3_z60__r10__cond__none",
        "dist_mr_k3_z60", 10, "none", "none", False, False,
    ),
    (
        "dist_mr_k3_z10__r10__cond__panic_10d_minus5",
        "dist_mr_k3_z10", 10, "none", "panic_10d_minus5", False, False,
    ),
    (
        "monoton_120d__r21__crash_10_5pct",
        "monoton_120d", 21, "crash_10_5pct", "none", False, False,
    ),
    (
        "resid_gap_reversion__r10__breadth_40",
        "resid_gap_reversion", 10, "breadth_40", "none", False, True,
    ),
]

SLEEVE_SHORT_LABELS = [
    "FactorMR_5d",
    "MR_5d_Breadth",
    "DistMR_z20_VolCon",
    "DistMR_z60_VolExp",
    "DistMR_z60",
    "DistMR_z10_Panic",
    "Monoton_120d",
    "ResidGap",
]

print(f"SP500 universe: {len(SP500)} tickers")
print(f"IS period:  {START_DATE} to {IS_END}")
print(f"OOS period: {OOS_SPLIT.date()} to {OOS_END}")
print(f"Output dir: {OUT_DIR}")


## Section 1: Data Download


In [ ]:
def safe_download(tickers, start, end, interval="1d"):
    """Download price data in chunks to avoid yfinance limits."""
    chunk_size = 25
    chunks = [tickers[i:i+chunk_size] for i in range(0, len(tickers), chunk_size)]
    close_frames, open_frames, vol_frames = [], [], []
    for chunk in chunks:
        try:
            data = yf.download(
                chunk, start=start, end=end, interval=interval,
                auto_adjust=True, progress=False, threads=False,
            )
            if data.empty:
                continue
            if isinstance(data.columns, pd.MultiIndex):
                c = data["Close"] if "Close" in data.columns.get_level_values(0) else data.iloc[:, :len(chunk)]
                o = data["Open"]  if "Open"  in data.columns.get_level_values(0) else c
                v = data["Volume"]if "Volume"in data.columns.get_level_values(0) else c * 0
            else:
                c = data[["Close"]].rename(columns={"Close": chunk[0]}) if len(chunk)==1 else data.get("Close", data)
                o = data[["Open"]].rename(columns={"Open": chunk[0]})   if len(chunk)==1 else data.get("Open",  data)
                v = data[["Volume"]].rename(columns={"Volume": chunk[0]})if len(chunk)==1 else data.get("Volume",data*0)
            for frame_list, frame in [(close_frames,c),(open_frames,o),(vol_frames,v)]:
                if isinstance(frame, pd.Series):
                    frame = frame.to_frame(name=chunk[0])
                frame_list.append(frame)
        except Exception as e:
            print(f"  Chunk failed ({chunk[:2]}...): {e}")

    close = pd.concat(close_frames, axis=1).sort_index().dropna(axis=1, how="all")
    open_ = pd.concat(open_frames,  axis=1).sort_index().reindex(columns=close.columns)
    vol   = pd.concat(vol_frames,   axis=1).sort_index().reindex(columns=close.columns)
    returns     = close.pct_change().fillna(0.0)
    log_returns = np.log(close / close.shift(1)).fillna(0.0)
    return {"close": close, "open": open_, "volume": vol,
            "returns": returns, "log_returns": log_returns}


In [ ]:
print("Downloading SP500 universe (IS: 2015-2023)...")
universe_is = safe_download(SP500, START_DATE, IS_END)
print(f"  Close shape: {universe_is['close'].shape}")

print("Downloading benchmark + sector ETFs (IS)...")
factor_tickers   = [BENCHMARK] + SECTOR_ETFS
factors_is       = safe_download(factor_tickers, START_DATE, IS_END)
benchmark_is     = factors_is["returns"].get(BENCHMARK, universe_is["returns"].mean(axis=1))
factor_returns_is= factors_is["returns"].reindex(columns=FACTORS)
print(f"  Factor shape: {factor_returns_is.shape}")


In [ ]:
# ── Sector map via yfinance (threaded) ────────────────────────────────────────
SECTOR_CACHE_PATH = os.path.join(OUT_DIR, "sector_map_cache.json")

def fetch_sector(ticker):
    try:
        info = yf.Ticker(ticker).info
        return ticker, info.get("sector", "Unknown")
    except Exception:
        return ticker, "Unknown"

def get_sector_map(tickers):
    cached = {}
    if os.path.exists(SECTOR_CACHE_PATH):
        with open(SECTOR_CACHE_PATH) as f:
            cached = json.load(f)
    missing = [t for t in tickers if t not in cached]
    if missing:
        print(f"  Fetching sector info for {len(missing)} tickers (threaded)...")
        with ThreadPoolExecutor(max_workers=20) as ex:
            futures = {ex.submit(fetch_sector, t): t for t in missing}
            for fut in as_completed(futures):
                t, s = fut.result()
                cached[t] = s
        with open(SECTOR_CACHE_PATH, "w") as f:
            json.dump(cached, f)
    return {t: cached.get(t, "Unknown") for t in tickers}

avail_tickers = list(universe_is["close"].columns)
print("Fetching sector map...")
sector_map_raw  = get_sector_map(avail_tickers)
sector_etf_map  = {t: GICS_TO_ETF.get(s, "XLK") for t, s in sector_map_raw.items()}
print(f"  Sector map: {len(sector_etf_map)} tickers")
print(Counter(sector_map_raw.values()).most_common(6))


In [ ]:
# ── Distance partners (top-3 by normalised price correlation) ─────────────────
print("Computing distance partners...")
log_ret_is  = universe_is["log_returns"]
log_price   = log_ret_is.cumsum()
norm_price  = (log_price - log_price.mean()) / log_price.std().clip(lower=1e-8)
corr_mat    = norm_price.corr()
dist_mat    = 1 - corr_mat

partners = {}
for ticker in dist_mat.columns:
    row = dist_mat[ticker].drop(ticker)
    partners[ticker] = row.nsmallest(3).index.tolist()

print(f"  Partners computed for {len(partners)} tickers")
print(f"  Example — AAPL peers: {partners.get('AAPL', [])}")


## Section 2: Inlined Pipeline Infrastructure


In [ ]:
# ── Metrics ────────────────────────────────────────────────────────────────────
def sharpe_ratio(returns, periods=252):
    s = returns.std()
    return returns.mean() / s * np.sqrt(periods) if s > 0 else 0.0

def ann_return(returns, periods=252):
    n = len(returns)
    return (1 + returns).prod() ** (periods / n) - 1 if n > 0 else 0.0

def ann_vol(returns, periods=252):
    return returns.std() * np.sqrt(periods)

def drawdown_series(returns):
    cum = (1 + returns).cumprod()
    return cum / cum.cummax() - 1

def max_drawdown(returns):
    return drawdown_series(returns).min()

def max_drawdown_duration(returns):
    dd = drawdown_series(returns)
    in_dd = (dd < 0).astype(int)
    max_dur, cur_dur = 0, 0
    for v in in_dd:
        cur_dur = cur_dur + 1 if v else 0
        max_dur = max(max_dur, cur_dur)
    return max_dur

def compute_turnover(positions):
    return positions.fillna(0.0).diff().abs().sum(axis=1)

def performance_summary(returns, positions=None, benchmark=None):
    sr    = sharpe_ratio(returns)
    ar    = ann_return(returns)
    av    = ann_vol(returns)
    mdd   = max_drawdown(returns)
    mdd_d = max_drawdown_duration(returns)
    to    = compute_turnover(positions).mean() if positions is not None else np.nan
    bmark_corr = np.nan
    ir         = np.nan
    if benchmark is not None:
        common = returns.index.intersection(benchmark.index)
        if len(common) > 30:
            r = returns.loc[common]
            b = benchmark.loc[common]
            bmark_corr = r.corr(b)
            ir = sharpe_ratio(r - b)
    return {
        "sharpe": sr, "ann_return": ar, "ann_vol": av,
        "max_drawdown": mdd, "max_drawdown_duration": mdd_d,
        "avg_daily_turnover": to, "benchmark_corr": bmark_corr,
        "information_ratio": ir,
    }


In [ ]:
# ── Liquidity filter ───────────────────────────────────────────────────────────
def liquidity_mask(close, volume, top_n=300, window=60):
    dv = (close * volume).rolling(window).mean()
    return dv.rank(axis=1, ascending=False) <= top_n

# ── Long/short position builder ────────────────────────────────────────────────
def build_long_short(signal, n_long=20, n_short=20):
    rank       = signal.rank(axis=1, ascending=False, na_option="bottom")
    n_trade    = signal.count(axis=1)
    long_mask  = rank <= n_long
    short_cut  = n_trade - (n_short - 1)
    short_mask = rank.ge(short_cut.values[:, None]) & signal.notna()
    pos        = long_mask.astype(float) - short_mask.astype(float)
    abs_sum    = pos.abs().sum(axis=1).replace(0, float("nan"))
    return pos.div(abs_sum, axis=0)

# ── Rebalance ──────────────────────────────────────────────────────────────────
def rebalance(positions, every=1):
    if every == 1:
        return positions
    mask = np.arange(len(positions)) % every == 0
    result = positions.copy()
    result.iloc[~mask] = np.nan
    return result.ffill()

# ── Backtest engine ────────────────────────────────────────────────────────────
def backtest(positions, returns):
    ret_al = returns.reindex(columns=positions.columns)
    return (positions.shift(1) * ret_al).sum(axis=1)

# ── Transaction costs ──────────────────────────────────────────────────────────
def apply_costs(gross_returns, positions, cost_bps):
    to = compute_turnover(positions).reindex(gross_returns.index).fillna(0.0)
    return gross_returns - to * (cost_bps / 10_000)


In [ ]:
# ── BarraLiteFactorModel ───────────────────────────────────────────────────────
class BarraLiteFactorModel:
    """
    Rolling-beta market factor + static sector dummies.
    Daily cross-sectional OLS to residualise returns.
    """
    def __init__(self, sector_map, window=252):
        self.sector_map = sector_map
        self.window     = window
        self._fitted    = False

    def fit(self, returns, benchmark, close):
        tickers = list(returns.columns)
        self._tickers = tickers
        bm = benchmark.reindex(returns.index).fillna(0.0)
        roll_cov = returns.rolling(self.window).cov(bm)
        roll_var = bm.rolling(self.window).var().replace(0, np.nan)
        self._betas = (roll_cov.div(roll_var, axis=0)).shift(1).fillna(1.0).clip(-3, 3)
        sectors = [self.sector_map.get(t, "Unknown") for t in tickers]
        unique_s = sorted(set(sectors))
        ref_s = Counter(sectors).most_common(1)[0][0]
        dum_s = [s for s in unique_s if s != ref_s]
        self._sector_dummies = pd.DataFrame(
            {s: [1.0 if sec == s else 0.0 for sec in sectors] for s in dum_s},
            index=tickers,
        )
        self._fitted = True
        return self

    def residualize(self, returns):
        if not self._fitted:
            raise RuntimeError("Call fit() first.")
        tickers = [t for t in self._tickers if t in returns.columns]
        ret = returns[tickers]
        betas = self._betas[tickers]
        sector_dum = self._sector_dummies.reindex(tickers)
        residuals = pd.DataFrame(index=ret.index, columns=tickers, dtype=float)
        for date in ret.index:
            r_t   = ret.loc[date]
            valid = r_t.dropna().index
            if len(valid) < 20:
                continue
            b_t = betas.loc[date, valid].values.reshape(-1, 1)
            sd  = sector_dum.reindex(valid).fillna(0.0).values
            X   = np.hstack([b_t, sd])
            if X[:, 0].std() > 1e-8:
                X[:, 0] = (X[:, 0] - X[:, 0].mean()) / X[:, 0].std()
            y = r_t[valid].values
            try:
                lr = LinearRegression(fit_intercept=True).fit(X, y)
                residuals.loc[date, valid] = y - lr.predict(X)
            except Exception:
                pass
        return residuals.astype(float)


In [ ]:
# ── ETF time-series residualization ───────────────────────────────────────────
def etf_residualize(returns, sector_etf_map, factor_returns):
    """Per-ticker OLS against its sector ETF; return residuals."""
    residuals = pd.DataFrame(index=returns.index, columns=returns.columns, dtype=float)
    for ticker in returns.columns:
        etf = sector_etf_map.get(ticker, "XLK")
        if etf not in factor_returns.columns:
            etf = factor_returns.columns[0]
        y = returns[ticker].fillna(0.0).values
        X = factor_returns[etf].reindex(returns.index).fillna(0.0).values.reshape(-1, 1)
        try:
            lr = LinearRegression(fit_intercept=True).fit(X, y)
            residuals[ticker] = y - lr.predict(X)
        except Exception:
            residuals[ticker] = returns[ticker]
    return residuals


In [ ]:
# ── Signal functions ───────────────────────────────────────────────────────────
def signal_mr_5d(returns, **kw):
    return -returns.rolling(5).mean()

def signal_monoton_120d(returns, **kw):
    mu = returns.rolling(120).mean()
    same_sign = (returns.gt(0) == mu.gt(0)).astype(float)
    return same_sign.rolling(120).mean() * mu.abs()

def signal_factor_model_resid_mr_5d(residual_returns, **kw):
    return -residual_returns.rolling(5).mean()

def signal_dist_mr(returns, zw, partners, **kw):
    price = (1 + returns).cumprod()
    norm  = price.div(price.iloc[0].clip(lower=1e-8), axis=1)
    spread = pd.DataFrame(index=returns.index, columns=returns.columns, dtype=float)
    for ticker in returns.columns:
        peers = [p for p in partners.get(ticker, []) if p in returns.columns]
        if peers:
            spread[ticker] = norm[ticker] - norm[peers].mean(axis=1)
    mu    = spread.rolling(zw).mean()
    sigma = spread.rolling(zw).std().clip(lower=1e-8)
    return -((spread - mu) / sigma).clip(-2, 2)

def signal_resid_gap_reversion(etf_residuals, **kw):
    return -etf_residuals.shift(1)

# ── Conditioning filters ───────────────────────────────────────────────────────
def apply_conditioning_filter(signal, filter_tag, returns, benchmark,
                               residual_returns=None):
    if filter_tag == "none":
        return signal
    gate = pd.Series(True, index=signal.index)
    if filter_tag == "residual_dispersion_high_20_q75":
        if residual_returns is not None:
            disp   = residual_returns.rolling(20).std().mean(axis=1)
            thresh = disp.rolling(252).quantile(0.75)
            gate   = disp >= thresh
    elif filter_tag == "breadth_weak_40":
        cum = (1 + returns).cumprod()
        pct_above = (cum > cum.rolling(200).mean()).mean(axis=1)
        gate = pct_above < 0.40
    elif filter_tag == "vol_contraction_10_60":
        bm = benchmark.reindex(returns.index).fillna(0.0)
        gate = bm.rolling(10).std() < bm.rolling(60).std()
    elif filter_tag == "vol_expansion_10_60":
        bm = benchmark.reindex(returns.index).fillna(0.0)
        gate = bm.rolling(10).std() > bm.rolling(60).std()
    elif filter_tag == "panic_10d_minus5":
        bm = benchmark.reindex(returns.index).fillna(0.0)
        cum10 = (1 + bm).rolling(10).apply(np.prod, raw=True) - 1
        gate  = cum10 < -0.05
    elif filter_tag == "breadth_40":
        cum = (1 + returns).cumprod()
        pct_above = (cum > cum.rolling(200).mean()).mean(axis=1)
        gate = pct_above < 0.40
    gate = gate.reindex(signal.index).fillna(False)
    return signal.where(gate.values[:, None], np.nan)

# ── Risk scalers ───────────────────────────────────────────────────────────────
def apply_risk_scaler(positions, scaler_tag, returns, benchmark):
    if scaler_tag == "none":
        return positions
    scale = pd.Series(1.0, index=positions.index)
    if scaler_tag == "trend_20_100":
        bm = benchmark.reindex(positions.index).fillna(0.0)
        ma_f = bm.rolling(20).mean()
        ma_s = bm.rolling(100).mean()
        scale = pd.Series(np.where(ma_f > ma_s, 0.25, 1.0), index=positions.index)
    elif scaler_tag == "trend_50_200":
        bm  = benchmark.reindex(positions.index).fillna(0.0)
        eq  = (1 + bm).cumprod()
        ma_f = eq.rolling(50).mean()
        ma_s = eq.rolling(200).mean()
        scale = pd.Series(np.where(ma_f > ma_s, 0.25, 1.0), index=positions.index)
    elif scaler_tag == "crash_10_5pct":
        bm   = benchmark.reindex(positions.index).fillna(0.0)
        c10d = (1 + bm).rolling(10).apply(np.prod, raw=True) - 1
        scale = pd.Series(np.where(c10d > 0.05, 0.25, 1.0), index=positions.index)
    elif scaler_tag == "breadth_40":
        cum = (1 + returns).cumprod()
        pct = (cum > cum.rolling(200).mean()).mean(axis=1)
        scale = pd.Series(np.where(pct < 0.40, 0.25, 1.0), index=positions.index)
    return positions.mul(scale.shift(1).fillna(1.0), axis=0)

# ── Equity-curve regime scale ──────────────────────────────────────────────────
def equity_curve_regime_scale(positions, returns, tradeable_mask=None):
    if tradeable_mask is not None:
        raw = (positions.shift(1) * returns.where(tradeable_mask)).sum(axis=1)
    else:
        raw = (positions.shift(1) * returns).sum(axis=1)
    equity = (1 + raw).cumprod()
    scale  = pd.Series(
        np.where(equity > equity.rolling(20).mean(), 1.0, 0.25),
        index=equity.index,
    )
    return positions.mul(scale.shift(1).fillna(1.0), axis=0)


In [ ]:
# ── run_sleeve() ───────────────────────────────────────────────────────────────
def run_sleeve(data, label, signal_name, rebal_every, risk_scaler_tag,
               cond_filter_tag, use_factor_model, use_etf_resid,
               benchmark_series, factor_rets, sector_etf_map_,
               sector_map_, partners_, n_long=N_LONG, n_short=N_SHORT):
    """Run one sleeve end-to-end; returns gross returns (costs applied at portfolio level)."""
    close   = data["close"]
    returns = data["returns"]
    volume  = data["volume"]

    liq_mask = liquidity_mask(close, volume, top_n=300, window=60)

    residual_returns = None
    etf_residuals    = None

    if use_factor_model:
        fm = BarraLiteFactorModel(sector_map=sector_map_, window=252)
        fm.fit(returns, benchmark_series, close)
        residual_returns = fm.residualize(returns)

    if use_etf_resid:
        etf_residuals = etf_residualize(returns, sector_etf_map_, factor_rets)

    # Signal
    if signal_name == "mr_5d":
        sig = signal_mr_5d(returns)
    elif signal_name == "monoton_120d":
        sig = signal_monoton_120d(returns)
    elif signal_name == "factor_model_resid_mr_5d":
        rr  = residual_returns if residual_returns is not None else returns
        sig = signal_factor_model_resid_mr_5d(rr)
    elif signal_name == "dist_mr_k3_z20":
        sig = signal_dist_mr(returns, zw=20, partners=partners_)
    elif signal_name == "dist_mr_k3_z60":
        sig = signal_dist_mr(returns, zw=60, partners=partners_)
    elif signal_name == "dist_mr_k3_z10":
        sig = signal_dist_mr(returns, zw=10, partners=partners_)
    elif signal_name == "resid_gap_reversion":
        er  = etf_residuals if etf_residuals is not None else returns
        sig = signal_resid_gap_reversion(er)
    else:
        raise ValueError(f"Unknown signal: {signal_name}")

    sig = sig.where(liq_mask.reindex_like(sig), np.nan)
    sig = apply_conditioning_filter(sig, cond_filter_tag, returns, benchmark_series, residual_returns)

    pos = build_long_short(sig, n_long=n_long, n_short=n_short)
    pos = equity_curve_regime_scale(pos, returns, tradeable_mask=liq_mask.reindex_like(pos))
    pos = apply_risk_scaler(pos, risk_scaler_tag, returns, benchmark_series)
    pos = rebalance(pos, every=rebal_every)

    gross   = backtest(pos, returns)
    metrics = performance_summary(gross, positions=pos, benchmark=benchmark_series)

    return {"returns": gross, "positions": pos, "metrics": metrics, "label": label}


In [ ]:
# ── Portfolio combination & weighting utilities ────────────────────────────────
def combine_sleeves(sleeve_results, weights, cost_bps, universe_returns):
    labels      = list(sleeve_results.keys())
    all_dates   = sleeve_results[labels[0]]["positions"].index
    all_tickers = sorted({t for res in sleeve_results.values() for t in res["positions"].columns})

    combined_pos = pd.DataFrame(0.0, index=all_dates, columns=all_tickers)
    for lbl, res in sleeve_results.items():
        w = weights.get(lbl, 0.0)
        aligned      = res["positions"].reindex(index=all_dates, columns=all_tickers).fillna(0.0)
        combined_pos += w * aligned

    ret_al       = universe_returns.reindex(index=all_dates, columns=all_tickers)
    gross_ret    = backtest(combined_pos, ret_al)
    net_ret      = apply_costs(gross_ret, combined_pos, cost_bps)
    sleeve_rets  = pd.DataFrame({lbl: res["returns"] for lbl, res in sleeve_results.items()})
    return net_ret, gross_ret, combined_pos, sleeve_rets


def equal_vol_weights(sleeve_results):
    vols     = {l: res["returns"].std() * np.sqrt(252) for l, res in sleeve_results.items()}
    inv_vols = {l: 1.0 / max(v, 1e-6) for l, v in vols.items()}
    total    = sum(inv_vols.values())
    return {l: iv / total for l, iv in inv_vols.items()}

def equal_weights(sleeve_results):
    n = len(sleeve_results)
    return {l: 1.0/n for l in sleeve_results}

def sharpe_weights(sleeve_results):
    sharpes = {l: max(sharpe_ratio(res["returns"]), 0.0) for l, res in sleeve_results.items()}
    total   = sum(sharpes.values())
    if total == 0:
        return equal_weights(sleeve_results)
    return {l: s/total for l, s in sharpes.items()}

def ridge_mv_weights(sleeve_results, alpha=1.0):
    ret_df = pd.DataFrame({l: res["returns"] for l, res in sleeve_results.items()}).dropna()
    mu     = ret_df.mean().values
    cov    = ret_df.cov().values
    n      = len(mu)
    reg    = cov + alpha * np.eye(n) * np.diag(cov).mean()
    try:
        inv_cov = np.linalg.inv(reg)
        w_raw   = np.clip(inv_cov @ mu, 0, None)
        total   = w_raw.sum()
        if total <= 0:
            return equal_weights(sleeve_results)
        w = w_raw / total
    except Exception:
        return equal_weights(sleeve_results)
    return {l: float(w[i]) for i, l in enumerate(sleeve_results)}

def vol_target_scale(portfolio_returns, target_vol, window=63, max_lev=5.0):
    rv    = portfolio_returns.rolling(window).std() * np.sqrt(252)
    scale = (target_vol / rv).clip(0, max_lev).shift(1).fillna(1.0)
    return portfolio_returns * scale


## Section 3: Run All 8 Sleeves (IS Period)


In [ ]:
print("Running all 8 sleeves on IS data (2015–2023)...")
print("(Factor model residualisation is the slowest step — expect ~5-15 min total)")
print()

sleeve_results_is = {}

for cfg in SLEEVE_CONFIGS:
    label, sig_name, rebal_ev, risk_tag, cond_tag, use_fm, use_etf = cfg
    print(f"  Running: {label[:65]}...")
    res = run_sleeve(
        data            = universe_is,
        label           = label,
        signal_name     = sig_name,
        rebal_every     = rebal_ev,
        risk_scaler_tag = risk_tag,
        cond_filter_tag = cond_tag,
        use_factor_model= use_fm,
        use_etf_resid   = use_etf,
        benchmark_series= benchmark_is,
        factor_rets     = factor_returns_is,
        sector_etf_map_ = sector_etf_map,
        sector_map_     = sector_map_raw,
        partners_       = partners,
    )
    sleeve_results_is[label] = res
    m = res["metrics"]
    print(f"    Sharpe={m['sharpe']:.2f}  AnnRet={m['ann_return']:.1%}  "
          f"AnnVol={m['ann_vol']:.1%}  MDD={m['max_drawdown']:.1%}")

print("\nAll 8 sleeves complete.")


In [ ]:
# Metrics summary table
rows = []
for short, (lbl, *_) in zip(SLEEVE_SHORT_LABELS, SLEEVE_CONFIGS):
    m = sleeve_results_is[lbl]["metrics"]
    rows.append({
        "Sleeve": short,
        "Sharpe":     round(m["sharpe"], 2),
        "Ann Return": f"{m['ann_return']:.1%}",
        "Ann Vol":    f"{m['ann_vol']:.1%}",
        "Max DD":     f"{m['max_drawdown']:.1%}",
        "Avg TO":     f"{m['avg_daily_turnover']:.3f}",
    })
metrics_df = pd.DataFrame(rows).set_index("Sleeve")
print("Sleeve Metrics (IS, Gross):")
display(metrics_df)


## Section 4: Figure 1 — Sleeve Return Correlation Heatmap


In [ ]:
sleeve_ret_df = pd.DataFrame({
    short: sleeve_results_is[lbl]["returns"]
    for short, (lbl, *_) in zip(SLEEVE_SHORT_LABELS, SLEEVE_CONFIGS)
}).dropna()

corr = sleeve_ret_df.corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f", cmap="RdYlGn",
    vmin=-0.5, vmax=0.5, center=0, linewidths=0.5,
    square=True, ax=ax, annot_kws={"size": 8},
)
ax.set_title("Figure 1: Sleeve Return Correlation Heatmap (IS, 2015–2023)",
             fontsize=12, pad=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right", fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "figure1_sleeve_correlation_heatmap.png"), dpi=150)
plt.show()
print(f"Saved: figure1_sleeve_correlation_heatmap.png")


## Section 5: Table 3 — Sleeve-by-Sleeve Portfolio Buildout


In [ ]:
buildout_rows   = []
current_labels  = []

for i, (short, cfg) in enumerate(zip(SLEEVE_SHORT_LABELS, SLEEVE_CONFIGS)):
    lbl = cfg[0]
    current_labels.append(lbl)
    sub = {l: sleeve_results_is[l] for l in current_labels}
    w   = equal_vol_weights(sub)
    net, gross, comb_pos, _ = combine_sleeves(sub, w, COST_BPS, universe_is["returns"])
    m   = performance_summary(net, positions=comb_pos, benchmark=benchmark_is)
    buildout_rows.append({
        "Step":        i + 1,
        "Added Sleeve":short,
        "Net Sharpe":  round(m["sharpe"], 2),
        "Ann Return":  f"{m['ann_return']:.1%}",
        "Ann Vol":     f"{m['ann_vol']:.1%}",
        "Max DD":      f"{m['max_drawdown']:.1%}",
        "Avg TO":      f"{m['avg_daily_turnover']:.3f}",
    })
    print(f"  Step {i+1}: +{short:22s}  Sharpe={m['sharpe']:.2f}  "
          f"AnnRet={m['ann_return']:.1%}  MDD={m['max_drawdown']:.1%}")

table3_df = pd.DataFrame(buildout_rows).set_index("Step")
display(table3_df)
table3_df.to_csv(os.path.join(OUT_DIR, "table3_portfolio_buildout.csv"))
print(f"\nSaved: table3_portfolio_buildout.csv")


## Section 6: Figure 2 — Robustness Heatmap


In [ ]:
all_labels   = [cfg[0] for cfg in SLEEVE_CONFIGS]
sharpe_order = sorted(all_labels,
    key=lambda l: sleeve_results_is[l]["metrics"]["sharpe"], reverse=True)

variants = {
    "All 8 sleeves":   all_labels,
    "MR sleeves only": all_labels[:6],
    "Momentum+Resid":  all_labels[6:],
    "Top 4 Sharpe":    sharpe_order[:4],
    "Drop worst":      sharpe_order[:-1],
    "Half (fixed)":    all_labels[::2],
}

weighting_schemes = {
    "Equal":    equal_weights,
    "EqualVol": equal_vol_weights,
    "Sharpe":   sharpe_weights,
    "RidgeMV":  ridge_mv_weights,
}

rob_mat = pd.DataFrame(index=variants, columns=weighting_schemes, dtype=float)

for vname, vlbls in variants.items():
    sub = {l: sleeve_results_is[l] for l in vlbls if l in sleeve_results_is}
    if not sub:
        continue
    for wname, wfn in weighting_schemes.items():
        w   = wfn(sub)
        net, _, comb_pos, _ = combine_sleeves(sub, w, COST_BPS, universe_is["returns"])
        rob_mat.loc[vname, wname] = round(sharpe_ratio(net), 2)
    print(f"  {vname}: {dict(rob_mat.loc[vname])}")

display(rob_mat)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    rob_mat.astype(float), annot=True, fmt=".2f", cmap="YlOrRd",
    vmin=0, linewidths=0.5, ax=ax, annot_kws={"size": 10},
)
ax.set_title("Figure 2: Robustness Heatmap — Net Sharpe by Portfolio Variant × Weighting",
             fontsize=10, pad=10)
ax.set_xlabel("Weighting Scheme")
ax.set_ylabel("Portfolio Variant")
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "figure2_robustness_heatmap.png"), dpi=150)
plt.show()
print("Saved: figure2_robustness_heatmap.png")


## Section 7: Table 5 — Equal-Vol Attribution


In [ ]:
ev_weights = equal_vol_weights(sleeve_results_is)
net_ev, gross_ev, comb_pos_ev, sleeve_ret_matrix = combine_sleeves(
    sleeve_results_is, ev_weights, COST_BPS, universe_is["returns"]
)

ret_df   = sleeve_ret_matrix.dropna()
cov_mat  = ret_df.cov().values
lbl_ord  = list(sleeve_results_is.keys())
w_vec    = np.array([ev_weights[l] for l in lbl_ord])
port_var = float(w_vec @ cov_mat @ w_vec)
port_mu  = float((ret_df * pd.Series(ev_weights, index=ret_df.columns)).sum(axis=1).mean() * 252)

sigma_w = cov_mat @ w_vec
attr_rows = []
for i, (short, lbl) in enumerate(zip(SLEEVE_SHORT_LABELS, lbl_ord)):
    vc  = w_vec[i] * sigma_w[i] / port_var if port_var > 0 else np.nan
    sra = ret_df[lbl].mean() * 252 if lbl in ret_df.columns else np.nan
    rc  = w_vec[i] * sra / port_mu  if port_mu != 0 else np.nan
    eff = rc / vc if (not np.isnan(rc)) and vc and vc > 0 else np.nan
    attr_rows.append({
        "Sleeve":        short,
        "Weight":        f"{w_vec[i]:.3f}",
        "Return Contrib":f"{rc:.1%}" if not np.isnan(rc) else "N/A",
        "Var Contrib":   f"{vc:.1%}" if not np.isnan(vc) else "N/A",
        "Efficiency":    f"{eff:.2f}" if not np.isnan(eff) else "N/A",
    })

table5_df = pd.DataFrame(attr_rows).set_index("Sleeve")
print("Table 5: Equal-Vol Attribution")
display(table5_df)
table5_df.to_csv(os.path.join(OUT_DIR, "table5_sleeve_attribution.csv"))
m_ev = performance_summary(net_ev, positions=comb_pos_ev, benchmark=benchmark_is)
print(f"\nFull portfolio (equal-vol, net):  Sharpe={m_ev['sharpe']:.2f}  "
      f"AnnRet={m_ev['ann_return']:.1%}  AnnVol={m_ev['ann_vol']:.1%}  MDD={m_ev['max_drawdown']:.1%}")
print("Saved: table5_sleeve_attribution.csv")


## Section 8: Figure 3 — Transaction Cost Sensitivity


In [ ]:
cost_sweep = [2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 25]

# Pre-compute combined portfolio gross returns once (0 bps)
_, gross_port, comb_pos_all, _ = combine_sleeves(
    sleeve_results_is, ev_weights, 0.0, universe_is["returns"]
)
port_to = compute_turnover(comb_pos_all).reindex(gross_port.index).fillna(0.0)

sleeve_sharpes = {s: [] for s in SLEEVE_SHORT_LABELS}
port_sharpes   = []

for cost in cost_sweep:
    port_net = gross_port - port_to * (cost / 10_000)
    port_sharpes.append(sharpe_ratio(port_net))
    for short, (lbl, *_) in zip(SLEEVE_SHORT_LABELS, SLEEVE_CONFIGS):
        sg = sleeve_results_is[lbl]["returns"]
        sp = sleeve_results_is[lbl]["positions"]
        to = compute_turnover(sp).reindex(sg.index).fillna(0.0)
        sn = sg - to * (cost / 10_000)
        sleeve_sharpes[short].append(sharpe_ratio(sn))

fig, ax = plt.subplots(figsize=(10, 6))
colors  = plt.cm.tab10(np.linspace(0, 1, len(SLEEVE_SHORT_LABELS)))
for (lbl_s, sharpes), color in zip(sleeve_sharpes.items(), colors):
    ax.plot(cost_sweep, sharpes, "--", alpha=0.6, color=color, label=lbl_s, linewidth=1.5)
ax.plot(cost_sweep, port_sharpes, "k-", linewidth=2.5, label="Portfolio (equal-vol)")
ax.axhline(0, color="red", linewidth=0.8, linestyle=":")
ax.axvline(COST_BPS, color="gray", linewidth=1.0, linestyle="--", alpha=0.7,
           label=f"{COST_BPS:.0f} bps baseline")
ax.set_xlabel("Transaction Cost (bps)", fontsize=11)
ax.set_ylabel("Net Sharpe Ratio",        fontsize=11)
ax.set_title("Figure 3: Transaction Cost Sensitivity (IS, 2015–2023)", fontsize=12)
ax.legend(loc="upper right", fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "figure3_cost_sensitivity.png"), dpi=150)
plt.show()
print("Saved: figure3_cost_sensitivity.png")


## Section 9: Figure 4 — Target-Vol Equity Curves


In [ ]:
target_vols = [None, 0.05, 0.10, 0.15]
vol_labels  = ["Unlevered", "5% Target Vol", "10% Target Vol", "15% Target Vol"]
colors_tv   = ["black", "royalblue", "darkorange", "seagreen"]

fig = plt.figure(figsize=(12, 8))
gs  = gridspec.GridSpec(2, 1, height_ratios=[2.5, 1], hspace=0.08)
ax_eq = fig.add_subplot(gs[0])
ax_dd = fig.add_subplot(gs[1], sharex=ax_eq)

for tv, lbl, color in zip(target_vols, vol_labels, colors_tv):
    r   = net_ev if tv is None else vol_target_scale(net_ev, tv)
    eq  = (1 + r).cumprod()
    dd  = drawdown_series(r)
    ax_eq.plot(eq.index, eq.values, color=color, linewidth=1.6, label=lbl)
    ax_dd.fill_between(dd.index, dd.values, 0, alpha=0.35, color=color)
    ax_dd.plot(dd.index, dd.values, color=color, linewidth=0.8)

ax_eq.set_ylabel("Cumulative Return", fontsize=11)
ax_eq.set_title("Figure 4: Target-Vol Equity Curves (IS, 2015–2023)", fontsize=12)
ax_eq.legend(fontsize=9)
ax_eq.grid(True, alpha=0.3)
ax_eq.set_yscale("log")
ax_dd.set_ylabel("Drawdown", fontsize=11)
ax_dd.set_xlabel("Date", fontsize=11)
ax_dd.grid(True, alpha=0.3)
ax_dd.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
plt.setp(ax_eq.get_xticklabels(), visible=False)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "figure4_target_vol_curves.png"), dpi=150)
plt.show()
print("Saved: figure4_target_vol_curves.png")

print("\nTarget-Vol Summary:")
print(f"{'Label':20s}  {'Sharpe':>7}  {'AnnRet':>8}  {'AnnVol':>8}  {'MaxDD':>8}")
for tv, lbl in zip(target_vols, vol_labels):
    r = net_ev if tv is None else vol_target_scale(net_ev, tv)
    print(f"{lbl:20s}  {sharpe_ratio(r):7.2f}  {ann_return(r):8.1%}  "
          f"{ann_vol(r):8.1%}  {max_drawdown(r):8.1%}")


## Section 10: Table 7 + Figure 5 — Full History OOS Analysis (2015–2026)


In [ ]:
print("Downloading extended data (2015–2026-04-30)...")
universe_full = safe_download(SP500, START_DATE, OOS_END)
factors_full  = safe_download([BENCHMARK] + SECTOR_ETFS, START_DATE, OOS_END)
benchmark_full     = factors_full["returns"].get(BENCHMARK, universe_full["returns"].mean(axis=1))
factor_ret_full    = factors_full["returns"].reindex(columns=FACTORS)
print(f"  Full universe: {universe_full['close'].shape}")
print(f"  Dates: {universe_full['close'].index[0].date()} – {universe_full['close'].index[-1].date()}")


In [ ]:
# Recompute distance partners on full data
print("Recomputing distance partners on full data...")
log_ret_full    = universe_full["log_returns"]
log_price_full  = log_ret_full.cumsum()
norm_price_full = (log_price_full - log_price_full.mean()) / log_price_full.std().clip(lower=1e-8)
dist_mat_full   = 1 - norm_price_full.corr()

partners_full = {}
for ticker in dist_mat_full.columns:
    row = dist_mat_full[ticker].drop(ticker)
    partners_full[ticker] = row.nsmallest(3).index.tolist()
print(f"  Partners for {len(partners_full)} tickers")

# Update sector map for any new tickers
avail_full = list(universe_full["close"].columns)
sector_map_full  = get_sector_map(avail_full)
sector_etf_map_full = {t: GICS_TO_ETF.get(s, "XLK") for t, s in sector_map_full.items()}


In [ ]:
print("Running all 8 sleeves on full data (2015–2026)...")
sleeve_results_full = {}

for cfg in SLEEVE_CONFIGS:
    label, sig_name, rebal_ev, risk_tag, cond_tag, use_fm, use_etf = cfg
    print(f"  Running: {label[:65]}...")
    res = run_sleeve(
        data            = universe_full,
        label           = label,
        signal_name     = sig_name,
        rebal_every     = rebal_ev,
        risk_scaler_tag = risk_tag,
        cond_filter_tag = cond_tag,
        use_factor_model= use_fm,
        use_etf_resid   = use_etf,
        benchmark_series= benchmark_full,
        factor_rets     = factor_ret_full,
        sector_etf_map_ = sector_etf_map_full,
        sector_map_     = sector_map_full,
        partners_       = partners_full,
    )
    sleeve_results_full[label] = res
    m = res["metrics"]
    print(f"    Sharpe={m['sharpe']:.2f}  AnnRet={m['ann_return']:.1%}  MDD={m['max_drawdown']:.1%}")

print("\nAll sleeves (full period) complete.")


In [ ]:
# Equal-vol weights from IS portion only
is_mask = sleeve_results_full[all_labels[0]]["returns"].index < OOS_SPLIT
is_results_from_full = {
    lbl: {
        "returns":   res["returns"].loc[is_mask],
        "positions": res["positions"].loc[is_mask],
    }
    for lbl, res in sleeve_results_full.items()
}
ev_weights_full = equal_vol_weights(is_results_from_full)

net_full, gross_full, comb_pos_full, _ = combine_sleeves(
    sleeve_results_full, ev_weights_full, COST_BPS, universe_full["returns"]
)
net_vt10 = vol_target_scale(net_full, 0.10)

is_ret  = net_vt10.loc[net_vt10.index < OOS_SPLIT]
oos_ret = net_vt10.loc[net_vt10.index >= OOS_SPLIT]

print(f"IS  ({is_ret.index[0].date()} – {is_ret.index[-1].date()}):  "
      f"Sharpe={sharpe_ratio(is_ret):.2f}  AnnRet={ann_return(is_ret):.1%}  MDD={max_drawdown(is_ret):.1%}")
print(f"OOS ({oos_ret.index[0].date()} – {oos_ret.index[-1].date()}): "
      f"Sharpe={sharpe_ratio(oos_ret):.2f}  AnnRet={ann_return(oos_ret):.1%}  MDD={max_drawdown(oos_ret):.1%}")


In [ ]:
# ── Table 7: Annual Performance ────────────────────────────────────────────────
annual_rows = []
for year in range(2015, net_vt10.index.year.max() + 1):
    yr = net_vt10[net_vt10.index.year == year]
    if len(yr) < 20:
        continue
    annual_rows.append({
        "Year":       year,
        "Period":     "IS" if year <= 2023 else "OOS",
        "Ann Return": f"{ann_return(yr):.1%}",
        "Ann Vol":    f"{ann_vol(yr):.1%}",
        "Sharpe":     round(sharpe_ratio(yr), 2),
        "Max DD":     f"{max_drawdown(yr):.1%}",
    })

table7_df = pd.DataFrame(annual_rows).set_index("Year")
print("Table 7: Annual Performance (10% Vol Target, Equal-Vol Weighted)")
display(table7_df)
table7_df.to_csv(os.path.join(OUT_DIR, "table7_annual_performance.csv"))
print("Saved: table7_annual_performance.csv")


In [ ]:
# ── Figure 5: Full History OOS Summary ────────────────────────────────────────
rolling_sr = net_vt10.rolling(252).apply(
    lambda x: x.mean() / x.std() * np.sqrt(252) if x.std() > 0 else 0, raw=True
)
equity_full_ts = (1 + net_vt10).cumprod()
dd_full_ts     = drawdown_series(net_vt10)
bm_eq_full     = (1 + benchmark_full.reindex(net_vt10.index).fillna(0.0)).cumprod()

fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(3, 1, height_ratios=[2.5, 1, 1], hspace=0.08)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], sharex=ax1)
ax3 = fig.add_subplot(gs[2], sharex=ax1)

# Equity
ax1.plot(equity_full_ts.index, equity_full_ts.values, color="steelblue", linewidth=2,
         label="Portfolio (10% vol target)")
ax1.plot(bm_eq_full.index, bm_eq_full.values, color="gray", linewidth=1.2,
         alpha=0.7, label="SPY")
ax1.axvline(OOS_SPLIT, color="red", linewidth=1.2, linestyle="--", label="IS/OOS split")
ax1.fill_betweenx(
    [equity_full_ts.min() * 0.9, equity_full_ts.max() * 1.1],
    OOS_SPLIT, net_vt10.index[-1], alpha=0.04, color="red"
)
ax1.set_yscale("log")
ax1.set_ylabel("Cumulative Return", fontsize=11)
ax1.set_title("Figure 5: Full History OOS Analysis — 10% Vol Target Portfolio vs SPY",
              fontsize=12)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
y_pos = equity_full_ts.max() ** 0.85
ax1.text(pd.Timestamp("2019-01-01"), y_pos, "IS",  ha="center", fontsize=12,
         color="navy",    alpha=0.45)
ax1.text(pd.Timestamp("2025-01-01"), y_pos, "OOS", ha="center", fontsize=12,
         color="darkred", alpha=0.45)

# Drawdown
ax2.fill_between(dd_full_ts.index, dd_full_ts.values, 0, alpha=0.5, color="steelblue")
ax2.plot(dd_full_ts.index, dd_full_ts.values, color="steelblue", linewidth=0.8)
ax2.axvline(OOS_SPLIT, color="red", linewidth=1.2, linestyle="--")
ax2.set_ylabel("Drawdown", fontsize=11)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax2.grid(True, alpha=0.3)

# Rolling Sharpe
ax3.plot(rolling_sr.index, rolling_sr.values, color="darkorange", linewidth=1.5,
         label="Rolling 1Y Sharpe")
ax3.axhline(0, color="black", linewidth=0.8)
ax3.axvline(OOS_SPLIT, color="red", linewidth=1.2, linestyle="--")
ax3.set_ylabel("Rolling Sharpe", fontsize=11)
ax3.set_xlabel("Date", fontsize=11)
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)

plt.setp(ax1.get_xticklabels(), visible=False)
plt.setp(ax2.get_xticklabels(), visible=False)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "figure5_oos_summary.png"), dpi=150)
plt.show()
print("Saved: figure5_oos_summary.png")


In [ ]:
# ── Final output manifest ──────────────────────────────────────────────────────
print("=" * 65)
print("FINAL RESULTS SUMMARY (10% Vol Target, Equal-Vol Weighted)")
print("=" * 65)
print(f"{'Metric':<28}  {'IS (2015-2023)':>15}  {'OOS (2024+)':>12}")
print("-" * 60)
for lbl, fn in [("Sharpe Ratio", sharpe_ratio), ("Ann Return", ann_return),
                ("Ann Vol",      ann_vol),       ("Max Drawdown", max_drawdown)]:
    iv, ov = fn(is_ret), fn(oos_ret)
    if lbl == "Sharpe Ratio":
        print(f"{lbl:<28}  {iv:>15.2f}  {ov:>12.2f}")
    else:
        print(f"{lbl:<28}  {iv:>15.1%}  {ov:>12.1%}")
print("=" * 65)

print("\nOutput files:")
saved = [
    "figure1_sleeve_correlation_heatmap.png",
    "figure2_robustness_heatmap.png",
    "figure3_cost_sensitivity.png",
    "figure4_target_vol_curves.png",
    "figure5_oos_summary.png",
    "table3_portfolio_buildout.csv",
    "table5_sleeve_attribution.csv",
    "table7_annual_performance.csv",
    "sector_map_cache.json",
]
for fname in saved:
    path   = os.path.join(OUT_DIR, fname)
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  [{status}] {fname}")
